# Decomposing Claim-Settlement Variability Across an Insurer's Organizational Hierarchy with PROC NESTED

## Executive Summary

A property-and-casualty insurer wants to know **where** the inconsistency in claim settlements originates: is it driven by differences between geographic regions, between branch offices, between individual adjusters, or simply by claim-to-claim noise? This notebook builds a synthetic, fully nested book of auto-claim data (Region > Branch > Adjuster > claim) and uses **PROC NESTED** to perform a random-effects analysis of variance, estimating the variance component contributed by each level of the hierarchy and reporting each as a percent of total. The result tells management which organizational layer to target for settlement-consistency improvements.

## Data Sources

All data is generated inline by the first DATA step — no external files, no network. The design is **completely nested**: each branch belongs to exactly one region, each adjuster to exactly one branch, and each claim to exactly one adjuster.

| Dataset | Rows | Variable | Type | Role | Description |
|---------|------|----------|------|------|-------------|
| `claims` | 40 | `Region` | char | CLASS (level 1, outermost) | Geographic region (5 regions: NE, SE, MW, WC, SW) |
| | | `Branch` | char | CLASS (level 2, nested in Region) | Branch office code (2 branches per region) |
| | | `Adjuster` | char | CLASS (level 3, nested in Branch) | Claims adjuster ID (2 adjusters per branch) |
| | | `Settlement` | num | VAR (dependent) | Indemnity paid on the auto claim, in USD |
| | | `CycleDays` | num | VAR (dependent) | Days from first notice of loss to settlement |

Synthetic structure: 5 regions x 2 branches x 2 adjusters x 2 claims = 40 observations. A region random effect, a branch-within-region random effect, an adjuster-within-branch random effect, and a claim-level residual are layered additively via `rand('NORMAL', ...)` so the variance components are recoverable. The region effect is drawn with the largest standard deviation (2,200) so that *where* a claim is handled is the dominant driver. Seed fixed with `call streaminit(20260531)`. A compact, fully balanced design keeps every level of the hierarchy populated with real degrees of freedom.

# Decomposing Claim-Settlement Variability with PROC NESTED

When a property-and-casualty insurer reviews how much it pays to settle auto claims, it often finds wide variation. Some of that variation is unavoidable (every accident is different), but some of it reflects **organizational** factors — one region runs richer than another, a branch office is more generous, an individual adjuster is an outlier.

The data have a strictly **nested** (hierarchical) structure:

```
Region  ->  Branch (nested in Region)  ->  Adjuster (nested in Branch)  ->  individual claims
```

A branch belongs to exactly one region and an adjuster to exactly one branch, so the factors are nested rather than crossed. `PROC NESTED` performs a random-effects analysis of variance for exactly this design and estimates a **variance component** at each level, expressed as a percent of total. That percent breakdown answers the business question: *which layer of the organization should we target to make settlements more consistent?*

## Step 1 — Generate a synthetic, fully nested book of claims

We simulate 5 regions, each with 2 branches, each with 2 adjusters, each handling 2 claims (40 rows total). We build the response from additive random effects so that each level genuinely contributes variance:

- a **region** effect (largest spread — regions differ most),
- a **branch-within-region** effect,
- an **adjuster-within-branch** effect,
- and a **claim-level residual** (the within-adjuster noise).

`call streaminit` fixes the seed for reproducibility; every effect is drawn with `rand('NORMAL', mean, sd)`. The region effect uses the widest standard deviation so it should claim the largest share of total variance. We also generate a second response, `CycleDays`, that shares the same hierarchy so we can demonstrate the multi-response analysis later.

In [1]:
data claims;
   call streaminit(20260531);
   length Region $2 Branch $6 Adjuster $9;

   /* Region-level random effect: regions differ the most. */
   do r = 1 to 5;
      Region = scan('NE SE MW WC SW', r);
      RegionEffect  = rand('NORMAL', 0, 2200);
      RegionCycle   = rand('NORMAL', 0, 11);

      /* Branch nested within region. */
      do b = 1 to 2;
         Branch = catx('-', Region, put(b, z2.));
         BranchEffect = rand('NORMAL', 0, 700);
         BranchCycle  = rand('NORMAL', 0, 4);

         /* Adjuster nested within branch. */
         do a = 1 to 2;
            Adjuster = catx('-', Branch, put(a, z1.));
            AdjEffect = rand('NORMAL', 0, 450);
            AdjCycle  = rand('NORMAL', 0, 2.5);

            /* Individual claims handled by this adjuster. */
            do claim = 1 to 2;
               Settlement = 8500
                          + RegionEffect + BranchEffect + AdjEffect
                          + rand('NORMAL', 0, 1100);
               CycleDays  = 21
                          + RegionCycle + BranchCycle + AdjCycle
                          + rand('NORMAL', 0, 6);
               if Settlement < 0 then Settlement = 0;
               if CycleDays  < 1 then CycleDays  = 1;
               output;
            end;
         end;
      end;
   end;

   keep Region Branch Adjuster Settlement CycleDays;
run;

NOTE: DATA claims


NOTE: Wrote claims (40 rows, 5 columns).
NOTE: DATA elapsed:
  wall  0.01 seconds
  cpu   0.01 seconds


## Step 2 — Sort by the nesting hierarchy

`PROC NESTED` requires the input data to be **sorted by the CLASS variables in the order they will be listed** — outermost factor first. We sort by `Region Branch Adjuster` so the procedure can walk the hierarchy correctly.

In [2]:
proc sort data=claims;
   by Region Branch Adjuster;
run;

NOTE: PROC SORT data=claims

NOTE: Unlicensed mode - input limited to 100 observations.
NOTE: Read 40 rows from claims.
NOTE: Wrote claims (40 rows, 5 columns).
NOTE: PROC SORT statement used.


## Step 3 — Variance-components analysis of the settlement amount

The core analysis. We list the CLASS variables outermost-to-innermost (`Region Branch Adjuster`); the innermost replication — the individual claims — is automatically treated as the error term. `VAR Settlement` names the single response.

The `LABEL` statement supplies a friendlier display name for the response in the output banner. The output produces the coefficients of expected mean squares, the analysis-of-variance table (with an F test of each variance component against zero), and the **variance component** estimate at each level together with its **percent of total**.

In [3]:
title1 'Variance Components of Auto-Claim Settlements';
title2 'Region / Branch / Adjuster Random-Effects ANOVA';

proc nested data=claims;
   class Region Branch Adjuster;
   var Settlement;
   label Settlement = 'Indemnity Paid (USD)';
run;

                                     Variance Components of Auto-Claim Settlements                                      
                                    Region / Branch / Adjuster Random-Effects ANOVA                                     


                       Nested Random Effects Analysis of Variance

Nested Random Effects Analysis of Variance for Variable Indemnity Paid (USD)
Variance Source       DF  Sum of Squares       F Value      Pr > F  Error Term   Mean Square  Variance Component    Percent of Total    EMS Coef
REGION                 4  62114122.126866          6.71      0.0304    BRANCH  15528530.531717  1651824.844989             54.4057      8.0000
BRANCH                 5  11569658.859028          1.89      0.1829  ADJUSTER  2313931.771806   272682.828942              8.9813      4.0000
ADJUSTER              10  12232004.560376          1.22      0.3348     Error  1223200.456038   111582.782346              3.6752      2.0000
Error                 20  20000697.82690

NOTE: Option TITLE changed to Variance Components of Auto-Claim Settlements.
NOTE: Option TITLE2 changed to Region / Branch / Adjuster Random-Effects ANOVA.
NOTE: PROC NESTED nested ANOVA / variance components

NOTE: PROC NESTED statement used.


## Step 4 — Analyze two responses at once

Management also cares about **cycle time** — how long claims take to settle. We add `CycleDays` to the `VAR` list. With more than one dependent variable, `PROC NESTED` additionally reports an **analysis of covariation** showing how the two responses co-vary at each level of the hierarchy (for example, whether the regions that pay more also settle slower).

In [4]:
title1 'Settlement and Cycle-Time Variance Components';
title2 'Two Responses Across the Same Nested Hierarchy';

proc nested data=claims;
   class Region Branch Adjuster;
   var Settlement CycleDays;
   label Settlement = 'Indemnity Paid (USD)'
         CycleDays  = 'Days to Settle';
run;

                                     Settlement and Cycle-Time Variance Components                                      
                                     Two Responses Across the Same Nested Hierarchy                                     


                       Nested Random Effects Analysis of Variance

Nested Random Effects Analysis of Variance for Variable Indemnity Paid (USD)
Variance Source       DF  Sum of Squares       F Value      Pr > F  Error Term   Mean Square  Variance Component    Percent of Total    EMS Coef
REGION                 4  62114122.126866          6.71      0.0304    BRANCH  15528530.531717  1651824.844989             54.4057      8.0000
BRANCH                 5  11569658.859028          1.89      0.1829  ADJUSTER  2313931.771806   272682.828942              8.9813      4.0000
ADJUSTER              10  12232004.560376          1.22      0.3348     Error  1223200.456038   111582.782346              3.6752      2.0000
Error                 20  20000697.82690

NOTE: Option TITLE changed to Settlement and Cycle-Time Variance Components.
NOTE: Option TITLE2 changed to Two Responses Across the Same Nested Hierarchy.
NOTE: PROC NESTED nested ANOVA / variance components

NOTE: PROC NESTED statement used.


## Step 5 — Variance-only view with the AOV option

For a quick variance-components summary across both responses, the `AOV` option restricts the output to the analysis-of-variance statistics and **suppresses the analysis-of-covariation** section. This is the compact view a portfolio analyst would scan when they only need the per-level variance breakdown for each response, not the cross-response covariation.

In [5]:
title1 'Variance Components Only (AOV)';

proc nested data=claims aov;
   class Region Branch Adjuster;
   var Settlement CycleDays;
   label Settlement = 'Indemnity Paid (USD)'
         CycleDays  = 'Days to Settle';
run;

title;

                                             Variance Components Only (AOV)                                             
                                     Two Responses Across the Same Nested Hierarchy                                     


                       Nested Random Effects Analysis of Variance

Nested Random Effects Analysis of Variance for Variable Indemnity Paid (USD)
Variance Source       DF  Sum of Squares       F Value      Pr > F  Error Term   Mean Square  Variance Component    Percent of Total    EMS Coef
REGION                 4  62114122.126866          6.71      0.0304    BRANCH  15528530.531717  1651824.844989             54.4057      8.0000
BRANCH                 5  11569658.859028          1.89      0.1829  ADJUSTER  2313931.771806   272682.828942              8.9813      4.0000
ADJUSTER              10  12232004.560376          1.22      0.3348     Error  1223200.456038   111582.782346              3.6752      2.0000
Error                 20  20000697.82690

NOTE: Option TITLE changed to Variance Components Only (AOV).
NOTE: PROC NESTED nested ANOVA / variance components

NOTE: PROC NESTED statement used.


## Interpreting the results

The **Percent of Total** column in the analysis-of-variance table is the headline. Reading it top-to-bottom gives the share of total settlement variability attributable to each organizational layer. For the settlement amount the run produces:

- **Region — 54.4%** (Pr > F = 0.0304). The data were generated with the largest region-level spread, and the analysis recovers it: more than half of all settlement variability is *between* regions, statistically significant evidence that *where* a claim is handled is the dominant driver.
- **Branch (within Region) — 9.0%** (Pr > F = 0.18). A modest additional slice once you drill from region down to individual branch offices; not significant here.
- **Adjuster (within Branch) — 3.7%** (Pr > F = 0.33). Little variation is traceable to the individual adjuster in this book of business.
- **Error — 32.9%.** The residual claim-to-claim noise within an adjuster; this is the irreducible variation that no organizational lever can remove.

Each level also carries an **F test (Pr > F)** of the null hypothesis that its variance component is zero. The small Region p-value (0.0304) is statistical evidence of genuine systematic differences between regions, not sampling chance.

The cycle-time response tells a parallel story: **Region accounts for 45.8%** of the variation in days-to-settle (Pr > F = 0.0448, again significant), with Branch and Adjuster contributing single-digit shares and the residual carrying 40.1%. So both *how much* an insurer pays and *how long* it takes are governed first and foremost by region.

The two-response run adds an **analysis of covariation**. Here the Region level drives the cross-products, and the overall **correlation coefficient is -0.36** — a negative relationship: the regions that pay larger settlements tend to *close them faster*, not slower. That is a useful, non-obvious finding — the expensive regions are not the slow ones.

**Business takeaway:** because Region dominates the percent-of-total breakdown for both responses, the insurer should standardize settlement guidelines and authority limits *across regions* first — that is where the largest, statistically significant inconsistency lives — before investing in branch- or adjuster-level interventions. The negative settlement/cycle-time correlation means there is no single region that is both the most expensive and the slowest, so the two problems can be tackled with separate, region-targeted programs. PROC NESTED turns a vague sense of "our settlements are inconsistent" into an actionable, layer-by-layer attribution of where that inconsistency lives.